In [ ]:
%pip install --upgrade pip
#%pip install torchvision

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ----------------- ---------------------- 1.8/4.2 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 13.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Celda 1 — Verificación de entorno
import sys
import torch
 
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
 
for paquete in ["torchvision", "sklearn", "matplotlib"]:
    try:
        modulo = __import__(paquete)
        print(f"{paquete}: {modulo.__version__} ✓")
    except ImportError:
        print(f"{paquete} no instalado — ejecuta: pip install {paquete}")
 
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo a usar: {dispositivo}")

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
GPU disponible: False
torchvision: 0.28.0+cpu ✓
sklearn: 1.9.0 ✓
matplotlib: 3.11.1 ✓
Dispositivo a usar: cpu


In [ ]:
# Celda 2 — Cargar y explorar el dataset
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
 
nombres_clases = [
    "avión", "automóvil", "ave", "gato", "ciervo",
    "perro", "rana", "caballo", "barco", "camión",
]
 
transformacion_basica = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)), #
# media/std de CIFAR-10
])
 
train_full = datasets.CIFAR10(root="./data", train=True, download=True,
transform=transformacion_basica)
test_full = datasets.CIFAR10(root="./data", train=False, download=True,
transform=transformacion_basica)
 
# Subconjunto para que el laboratorio sea manejable en 2 horas
train_dataset = Subset(train_full, range(5000))
test_dataset = Subset(test_full, range(1000))
 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
 
print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")
print(f"Forma de una imagen: {train_full[0][0].shape}") # [canales, alto, ancho]
 
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    img, etiqueta = train_full[i]
    ax.imshow(img.permute(1, 2, 0) * 0.25 + 0.45) # desnormalizar aproximadamente
#para visualizar
    ax.set_title(nombres_clases[etiqueta])
    ax.axis("off")
plt.show()

13.4%

In [ ]:
# Celda 3 — Arquitectura CNN para imágenes a color
import torch.nn as nn
 
torch.manual_seed(42)
 
class CNNCifar(nn.Module):
    def __init__(self, n_clases=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3,
padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3,
padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, n_clases)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x))) # 32x32 -> 16x16, 32 canales
        x = self.pool(self.relu(self.conv2(x))) # 16x16 -> 8x8, 64 canales
        x = x.flatten(start_dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)
   
modelo_scratch = CNNCifar().to(dispositivo)
print(modelo_scratch)

In [ ]:
# Celda 4 — Entrenamiento desde cero
import torch.optim as optim
import time
 
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_scratch.parameters(), lr=0.001)
 
def entrenar_una_epoca(modelo, loader, criterio, optimizador, dispositivo):
    modelo.train()
    perdida_total = 0.0
 
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
        optimizador.zero_grad()
        salida = modelo(imagenes)
        perdida = criterio(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()
    return perdida_total / len(loader)
 
inicio = time.time()
for epoch in range(10):
    perdida = entrenar_una_epoca(modelo_scratch, train_loader, criterio, optimizador,
dispositivo)
    print(f"E   poch {epoch + 1} | Pérdida: {perdida:.4f}")
tiempo_scratch = time.time() - inicio
print(f"\nTiempo total de entrenamiento (desde cero): {tiempo_scratch:.1f} segundos")

In [ ]:
# Celda 5 — Preparar los datos para ResNet18 (espera entradas de 224x224)
transformacion_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #
# normalización de ImageNet
])
train_full_resnet = datasets.CIFAR10(root="./data", train=True, download=True,
transform=transformacion_resnet)
test_full_resnet = datasets.CIFAR10(root="./data", train=False, download=True,
transform=transformacion_resnet)
 
train_dataset_resnet = Subset(train_full_resnet, range(5000))
test_dataset_resnet = Subset(test_full_resnet, range(1000))
 
train_loader_resnet = DataLoader(train_dataset_resnet, batch_size=32, shuffle=True)
test_loader_resnet = DataLoader(test_dataset_resnet, batch_size=32, shuffle=False)

In [ ]:
# Celda 6 — Cargar ResNet18 preentrenada y congelar sus pesos (feature extraction)
import torchvision.models as models
 
modelo_transfer = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
 
for parametro in modelo_transfer.parameters():
    parametro.requires_grad = False # congelar TODA la red preentrenada
 
# Reemplazar la capa final por una nueva, entrenable, con 10 clases (CIFAR-10)
modelo_transfer.fc = nn.Linear(modelo_transfer.fc.in_features, 10)
modelo_transfer = modelo_transfer.to(dispositivo)
 
# Verificar cuántos parámetros son entrenables vs. totales
params_entrenables = sum(p.numel() for p in modelo_transfer.parameters() if
p.requires_grad)
params_totales = sum(p.numel() for p in modelo_transfer.parameters())
print(f"Parámetros entrenables: {params_entrenables:,} de {params_totales:,} totales")
 

In [ ]:
# Celda 7 — Entrenar solo la capa nueva (ResNet18)
optimizador_transfer = optim.Adam(modelo_transfer.fc.parameters(), lr=0.001)

inicio = time.time()
for epoch in range(10):
    perdida = entrenar_una_epoca(modelo_transfer, train_loader_resnet, criterio,
                                 optimizador_transfer, dispositivo)
    print(f"Epoch {epoch+1} | Pérdida: {perdida:.4f}")
tiempo_transfer = time.time() - inicio

print(f"\nTiempo total de entrenamiento (transfer learning): {tiempo_transfer:.1f} segundos")


In [ ]:
# Celda 8 — Evaluar ambos modelos en test y comparar
from sklearn.metrics import accuracy_score
 
def evaluar(modelo, loader, dispositivo):
    modelo.eval()
    predicciones, reales = [], []
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes = imagenes.to(dispositivo)
            salida = modelo(imagenes)
            pred = torch.argmax(salida, dim=1).cpu()
            predicciones.extend(pred.tolist())
            reales.extend(etiquetas.tolist())
    return accuracy_score(reales, predicciones)
 
acc_scratch = evaluar(modelo_scratch, test_loader, dispositivo)
acc_transfer = evaluar(modelo_transfer, test_loader_resnet, dispositivo)
 
print(f"{'Modelo':<28} {'Accuracy (test)':>15} {'Tiempo (s)':>12}")
print("-" * 57)
print(f"{'CNN desde cero':<28} {acc_scratch:>15.3f} {tiempo_scratch:>12.1f}")
print(f"{'Transfer Learning (ResNet18)':<28} {acc_transfer:>15.3f}{tiempo_transfer:>12.1f}")

In [ ]:
# BONO — Fine-tuning: descongelar el último bloque de ResNet18 y comparar
modelo_finetune = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for nombre, parametro in modelo_finetune.named_parameters():
    parametro.requires_grad = "layer4" in nombre or "fc" in nombre
 
modelo_finetune.fc = nn.Linear(modelo_finetune.fc.in_features, 10)
modelo_finetune = modelo_finetune.to(dispositivo)
 
optimizador_finetune = optim.Adam(
    filter(lambda p: p.requires_grad, modelo_finetune.parameters()),
    lr=0.0001, # LR bajo — ver Sección 6.6 del tema de teoría
)
# Repetir el ciclo de entrenamiento de la Celda 7 con este modelo
# y comparar accuracy contra la versión de feature extraction pura (Celda 6-8).